In [1]:
!pip install --upgrade pip --quiet
!pip install --upgrade diffusers transformers accelerate controlnet-aux datasets peft --quiet
!pip install torch-fidelity lpips --quiet
!pip install -q torch torchvision
!pip install -q safetensors datasets tqdm peft
!pip install scikit-image opencv-python-headless --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.0 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.

In [2]:
import torch
from torchvision import transforms
from diffusers import (
    DiffusionPipeline,
    StableDiffusionControlNetPipeline, 
    ControlNetModel, 
    AutoencoderKL, 
    DDPMScheduler,
    UNet2DConditionModel,
    UniPCMultistepScheduler
)
from diffusers.utils import load_image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
from PIL import Image
import numpy as np
import lpips
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast 
from torch_fidelity import calculate_metrics
from skimage.metrics import structural_similarity as ssim
import cv2
import warnings
from peft import LoraConfig, get_peft_model
warnings.filterwarnings("ignore")

2025-11-19 21:41:05.569206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763588465.720366      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763588465.768328      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

# Arguments

In [3]:
# Data
base_dir = "/kaggle/input/face-sketches-collection/small_hed-augmented_ffhq_dataset"
test_dir = base_dir

# Training and Testing
num_epochs = 20
batch_size = 4 
accumulation_steps = 8
effective_batch_size = batch_size * accumulation_steps
device = "cuda"
prompt = "a realistic photo of a human face"
controlnet_name = "lllyasviel/sd-controlnet-hed"
stable_diff_name = "botp/stable-diffusion-v1-5"
padding = "max_length"
return_tensors="pt"
scaler = GradScaler()
patience = 5  
best_eval_loss = float('inf')
best_model_path = "/kaggle/working/lora-controlnet_best_model"
latest_model_path = "/kaggle/working/lora-controlnet_latest_model"
generated_dir = "/kaggle/working/generated_for_metrics"
max_eval_samples = 500 

# Dataset

In [4]:
class SketchToPhotoDataset(Dataset):
    def __init__(self, hed_dir, photo_dir, max_samples=None):
        self.hed_dir = hed_dir
        self.photo_dir = photo_dir
        self.hed_files = sorted(os.listdir(hed_dir))
        self.photo_files = sorted(os.listdir(photo_dir))
        
        if max_samples and len(self.hed_files) > max_samples:
            import random
            indices = random.sample(range(len(self.hed_files)), max_samples)
            self.hed_files = [self.hed_files[i] for i in indices]
            self.photo_files = [self.photo_files[i] for i in indices]
        
        self.condition_transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor()  
        ])
        self.target_transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]) 
        ])
    
    def __len__(self):
        return len(self.hed_files)

    def __getitem__(self, idx):
        hed_path = os.path.join(self.hed_dir, self.hed_files[idx])
        photo_path = os.path.join(self.photo_dir, self.photo_files[idx])
        
        hed_image = Image.open(hed_path).convert("RGB")
        photo = Image.open(photo_path).convert("RGB")
        
        hed_image = self.condition_transform(hed_image)  
        photo = self.target_transform(photo) 
        
        return {"hed": hed_image, "photo": photo}

In [6]:
train_dataset = SketchToPhotoDataset(
    hed_dir=os.path.join(base_dir, "train", "sketches"),
    photo_dir=os.path.join(base_dir, "train", "photos"),
    max_samples=None 
)
val_dataset = SketchToPhotoDataset(
    hed_dir=os.path.join(base_dir, "val", "sketches"),
    photo_dir=os.path.join(base_dir, "val", "photos"),
    max_samples=None 
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Models

In [8]:
controlnet = ControlNetModel.from_pretrained(
    controlnet_name,
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    stable_diff_name, 
    controlnet=controlnet, 
    torch_dtype=torch.float16
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0", "conv1", "conv2","conv_in"],
    lora_dropout=0.1,
    bias="none",
)
# pipe.enable_xformers_memory_efficient_attention()

pipe.vae.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)
pipe.unet.requires_grad_(False)

pipe.controlnet = get_peft_model(pipe.controlnet, lora_config)
print("Trainable parameters: ", pipe.controlnet.print_trainable_parameters())

pipe.to(device) 

optimizer = torch.optim.AdamW(pipe.controlnet.parameters(), lr=2e-4, weight_decay=1e-2) 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

trainable params: 4,402,416 || all params: 365,681,536 || trainable%: 1.2039
Trainable parameters:  None


# Training

In [9]:
import json
training_logs = []
log_file_path = "/kaggle/working/training_logs.json"

In [ ]:
# patience_counter = 0

# for epoch in range(num_epochs):
#     pipe.controlnet.train()
#     epoch_loss = 0
#     epoch_log = {'epoch' : epoch + 1}
#     progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch} Training")
    
#     for step, batch in enumerate(progress_bar):
#         hed_images = batch["hed"].to(device)
#         photos = batch["photo"].to(device)
        
#         with autocast():
#             with torch.no_grad():
#                 latents = pipe.vae.encode(photos).latent_dist.sample() * pipe.vae.config.scaling_factor
            
#             # timesteps and noise
#             bsz = latents.shape[0]
#             timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=device).long()
#             noise = torch.randn_like(latents)
#             noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            
#             # Encode prompt
#             use_null_prompt = torch.rand(1).item() < 0.1
            
#             if use_null_prompt:
#                 final_prompt = "" 
#             else:
#                 final_prompt = prompt
                
#             text_inputs = pipe.tokenizer(
#                 final_prompt, 
#                 padding=padding, 
#                 max_length=pipe.tokenizer.model_max_length, 
#                 truncation=True, 
#                 return_tensors=return_tensors
#             )
        
#             text_input_ids = text_inputs.input_ids.to(device)
            
#             with torch.no_grad():
#                 encoder_hidden_states = pipe.text_encoder(text_input_ids)[0]
#                 if encoder_hidden_states.shape[0] != bsz:
#                     encoder_hidden_states = encoder_hidden_states.repeat(bsz, 1, 1)

#             # Forward ControlNet
#             controlnet_output = pipe.controlnet(
#                 sample=noisy_latents,
#                 timestep=timesteps,
#                 encoder_hidden_states=encoder_hidden_states,
#                 controlnet_cond=hed_images,
#                 return_dict=True 
#             )
            
#             down_block_res_samples = controlnet_output.down_block_res_samples
#             mid_block_res_sample = controlnet_output.mid_block_res_sample
            
#             # Forward UNet
#             noise_pred = pipe.unet(
#                 noisy_latents, 
#                 timestep=timesteps, 
#                 encoder_hidden_states=encoder_hidden_states, 
#                 down_block_additional_residuals=down_block_res_samples, 
#                 mid_block_additional_residual=mid_block_res_sample
#             ).sample
            
#             # loss 
#             loss = torch.nn.functional.mse_loss(noise_pred, noise)
#             epoch_loss += loss.item()
            
#             loss = loss / accumulation_steps
        
#         scaler.scale(loss).backward()
        
#         if (step + 1) % accumulation_steps == 0:
#             scaler.step(optimizer)
#             scaler.update() 
#             optimizer.zero_grad()
        
#         progress_bar.set_postfix(Loss=f"{loss.item() * accumulation_steps:.4f}")
    
#     avg_train_loss = epoch_loss / len(train_dataloader)
#     epoch_log['avg_loss'] = avg_train_loss
#     print(f"\nEpoch {epoch}, Avg Train Loss: {avg_train_loss:.4f}")

#     # Validation
#     pipe.controlnet.eval()
#     val_loss = 0
#     val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch} Validation")
    
#     with torch.no_grad():
#         for batch in val_progress_bar:
#             hed_images = batch["hed"].to(device)
#             photos = batch["photo"].to(device)
            
#             with autocast(): 
#                 latents = pipe.vae.encode(photos).latent_dist.sample() * pipe.vae.config.scaling_factor
#                 bsz = latents.shape[0]
#                 timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=device).long()
#                 noise = torch.randn_like(latents)
#                 noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
                
#                 text_inputs = pipe.tokenizer(
#                     prompt, 
#                     padding=padding, 
#                     max_length=pipe.tokenizer.model_max_length, 
#                     truncation=True, 
#                     return_tensors=return_tensors
#                 )
                
#                 text_input_ids = text_input_ids.to(device)
                
#                 encoder_hidden_states = pipe.text_encoder(text_input_ids)[0]
                
#                 if encoder_hidden_states.shape[0] != bsz:
#                     encoder_hidden_states = encoder_hidden_states.repeat(bsz, 1, 1)
                
#                 controlnet_output = pipe.controlnet(
#                     sample=noisy_latents,
#                     timestep=timesteps,
#                     encoder_hidden_states=encoder_hidden_states,
#                     controlnet_cond=hed_images,
#                     return_dict=True
#                 )
                
#                 noise_pred = pipe.unet(
#                     noisy_latents, 
#                     timestep=timesteps, 
#                     encoder_hidden_states=encoder_hidden_states, 
#                     down_block_additional_residuals=controlnet_output.down_block_res_samples, 
#                     mid_block_additional_residual=controlnet_output.mid_block_res_sample
#                 ).sample
                
#                 val_loss += torch.nn.functional.mse_loss(noise_pred, noise).item()
            
#             val_progress_bar.set_postfix(Val_Loss=f"{val_loss / len(val_dataloader):.4f}")
            
#     avg_val_loss = val_loss / len(val_dataloader)
#     print(f"Epoch {epoch}, Avg Val Loss: {avg_val_loss:.4f}")
#     training_logs.append(epoch_log)
#     try:
#         with open(log_file_path, 'w') as f:
#             json.dump(training_logs, f, indent=4)
#     except Exception as e:
#         print(f"Lỗi khi lưu log: {e}")
#     # Early stopping
#     if avg_val_loss < best_eval_loss:
#         best_eval_loss = avg_val_loss
#         patience_counter = 0
        
#         pipe.controlnet.save_pretrained(best_model_path)
#         print(f"Saved best model at: {best_model_path}")
        
#     else:
#         patience_counter += 1
#         print(f"Patience: {patience_counter} / {patience}")
        
#         if patience_counter >= patience:
#             print(f"Early stopping after {patience} epochs.")
#             break 
    
#     scheduler.step()

# pipe.controlnet.save_pretrained(best_model_path)
# print(f"Saved best model (Eval Loss: {best_eval_loss:.4f}) at: {best_model_path}")
# print(f"Saved final model at: {latest_model_path}")

In [15]:
# !zip -r -q /kaggle/working/controlnet_best_model.zip /kaggle/working/controlnet_best_model

# Testing

In [5]:
!pip install -q gdown

In [6]:
import gdown

url = 'https://drive.google.com/drive/folders/1LRoSH6IjILLgnTip2Jvg4-D_2H6j2SFO'

gdown.download_folder(url, output=best_model_path, quiet=True)

['/kaggle/working/lora-controlnet_best_model/adapter_config.json',
 '/kaggle/working/lora-controlnet_best_model/adapter_model.safetensors',
 '/kaggle/working/lora-controlnet_best_model/README.md']

In [7]:
real_dir = os.path.join(test_dir, "test", "photos")
sketch_dir = os.path.join(test_dir, "test", "sketches")

os.makedirs(generated_dir, exist_ok=True)

generator = torch.Generator(device=device).manual_seed(1234)

In [8]:
prompt = """(hyper-realistic photo:1.2), (ultra-detailed skin texture:1.1), 
            detailed pores, realistic eyes, sharp focus, 
            8k UHD, professional studio lighting, DSLR"""

negative_prompt = """(drawing:1.4), (sketch:1.4), (painting:1.3), cartoon, 3D, 
                    render, CGI, anime, illustration, (deformed:1.2), (disfigured:1.2), 
                    ugly, bad anatomy, (blurry:1.1), low quality, low-res"""

In [9]:
from peft import PeftModel

In [10]:
controlnet = ControlNetModel.from_pretrained(
    controlnet_name, 
    torch_dtype=torch.float16
)

controlnet = PeftModel.from_pretrained(controlnet , best_model_path)
controlnet = controlnet.merge_and_unload()
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    stable_diff_name, 
    controlnet=controlnet, 
    torch_dtype=torch.float16
)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/543 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

safety_checker/model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

...

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:03<00:00, 181MB/s] 


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


### LPIPS


In [11]:
lpips_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

total_lpips_distance = 0
image_count = 0

sketch_files = sorted(os.listdir(sketch_dir))

In [12]:
for i, filename in enumerate(tqdm(sketch_files)):
    if max_eval_samples and i >= max_eval_samples:
        break

    sketch_path = os.path.join(sketch_dir, filename)
    real_path = os.path.join(real_dir, filename)
    generated_path = os.path.join(generated_dir, filename)

    condition_image = load_image(sketch_path).resize((512, 512))

    generated_image_pil = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=condition_image,
        num_inference_steps=30,
        generator=generator,
        guidance_scale=7.5,
        controlnet_conditioning_scale=0.9
    ).images[0]
    
    generated_image_pil.save(generated_path)
    real_image_pil = load_image(real_path)
    real_tensor = lpips_transform(real_image_pil).to(device)
    gen_tensor = lpips_transform(generated_image_pil).to(device)

    with torch.no_grad():
        dist = loss_fn_vgg(real_tensor.unsqueeze(0), gen_tensor.unsqueeze(0))
    
    total_lpips_distance += dist.item()
    image_count += 1

  0%|          | 0/158 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  1%|          | 1/158 [00:12<32:43, 12.51s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

  1%|▏         | 2/158 [00:24<31:27, 12.10s/it]

...

 99%|█████████▊| 156/158 [30:42<00:23, 11.82s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

 99%|█████████▉| 157/158 [30:54<00:11, 11.82s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 158/158 [31:06<00:00, 11.81s/it]


In [21]:
avg_lpips = total_lpips_distance / image_count

print(f"Average LPIPS: {avg_lpips:.4f}")

Average LPIPS: 0.6728


### FID and KID

In [22]:
metrics = calculate_metrics(
    input1=real_dir,
    input2=generated_dir,
    cuda=True,
    fid=True,
    kid=True,
    input1_max_samples=image_count,
    input2_max_samples=image_count,
    kid_subset_size=image_count
)

print(f"FID: {metrics['frechet_inception_distance']:.4f}")
print(f"KID Mean: {metrics['kernel_inception_distance_mean']:.4f}")
print(f"KID Std: {metrics['kernel_inception_distance_std']:.4f}")

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:01<00:00, 92.1MB/s]
Extracting features from input1
Looking for samples non-recursivelty in "/kaggle/input/face-sketches-collection/small_hed-augmented_ffhq_dataset/test/photos" with extensions png,jpg,jpeg
Found 158 samples
Processing samples                                                        
Extracting features from input2
Looking for samples non-recursivelty in "/kaggle/working/generated_for_metrics" with extensions png,jpg,jpeg
Found 158 samples
Processing samples                                                        
Frechet Inception Distance: 205.00240507577774
                                                                                 

FID: 205.0024
KID Mean: 0.1624
KID Std: 0.0000


Kernel Inception Distance: 0.162406924330909 ± 1.8911465898682376e-07


### SSIM

In [13]:
grayscale_gen_dir = "/kaggle/working/grayscale_generated_dir"
grayscale_real_dir = "/kaggle/working/grayscale_real_dir"

os.makedirs(grayscale_gen_dir, exist_ok=True)
os.makedirs(grayscale_real_dir, exist_ok=True)

In [14]:
def calculate_ssim(real_path_source, gen_path_source):
    scores = []
    filenames = sorted(os.listdir(gen_path_source))
    
    for filename in tqdm(filenames, desc="Processing SSIM"):
        path_real = os.path.join(real_path_source, filename)
        path_gen = os.path.join(gen_path_source, filename)
        
        if os.path.exists(path_real) and os.path.exists(path_gen):
            img_real = cv2.imread(path_real)
            img_gen = cv2.imread(path_gen)
            
            if img_real is None or img_gen is None:
                continue
                
            if img_real.shape != img_gen.shape:
                img_real = cv2.resize(img_real, (img_gen.shape[1], img_gen.shape[0]))

            img_real_gray = cv2.cvtColor(img_real, cv2.COLOR_BGR2GRAY)
            img_gen_gray = cv2.cvtColor(img_gen, cv2.COLOR_BGR2GRAY)
            
            save_path_real_gray = os.path.join(grayscale_real_dir, filename)
            save_path_gen_gray = os.path.join(grayscale_gen_dir, filename)
            
            cv2.imwrite(save_path_real_gray, img_real_gray)
            cv2.imwrite(save_path_gen_gray, img_gen_gray)
            
            score = ssim(img_real_gray, img_gen_gray, data_range=255)
            scores.append(score)
            
    return np.mean(scores)

In [15]:
current_ssim = calculate_ssim(real_dir, generated_dir)

print(f"Average SSIM: {current_ssim:.4f}")

Processing SSIM: 100%|██████████| 158/158 [00:08<00:00, 18.59it/s]

Average SSIM: 0.2786


In [16]:
!zip -r -q /kaggle/working/generated_for_metrics.zip /kaggle/working/generated_for_metrics
!zip -r -q /kaggle/working/grayscale_generated_dir.zip /kaggle/working/grayscale_generated_dir
!zip -r -q /kaggle/working/grayscale_real_dir.zip /kaggle/working/grayscale_real_dir